# 输出实际用途与恢复门逐层核查
来源固定，模型冻结，只读账本；不移除门槛、不修改动作、不重跑收益。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import prepare_on_policy_recovery
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/output_routing_diagnosis_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
accounts,account_audit=_source(root,plan['account_source'],['inputs/frame',*[f'analysis/native/cash_start_2023_current_anchor_{c}' for c in plan['costs']]])
action,action_audit=_source(root,plan['action_source'],['inputs/plan','controller/source_recovery_selection']);model_source,model_audit=_source(root,plan['controller_source'],['inputs/controller'])
selection=action['controller/source_recovery_selection'];spec=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);model=model_source['inputs/controller']['frozen_recovery_model']
release=json.loads((root/'production/stage1_futu_sim_v1/runtime_release.json').read_text());live_summary=json.loads((root/release['action']['card_path']/'summary.json').read_text());assert hashlib.sha256((root/release['action']['card_path']/'summary.json').read_bytes()).hexdigest()==release['action']['summary_sha256']
assert live_summary['source_stage1']['selected_model']['minimum_source_alpha_probability']==selection['minimum_source_alpha_probability']
frame=accounts['inputs/frame'].set_index('trade_date',drop=False);funnel=[];records={}
for cost in plan['costs']:
 book=accounts[f'analysis/native/cash_start_2023_current_anchor_{cost}'];ledger=book['independent_replay'];previous_q=np.zeros(2);previous_cash=plan['initial_cash'];previous_nav=None;history=[];daily=[]
 for row in ledger.itertuples(index=False):
  source=frame.loc[pd.Timestamp(row.trade_date)];known=source[['known_soxl','known_soxx']].to_numpy(float);nav=float(previous_q@known+previous_cash)
  if previous_nav is not None:history.append(nav/previous_nav-1)
  previous_nav=nav;raw_beta=3*source.soxl_weight+source.soxx_weight;target=min(raw_beta,selection['action_beta'])
  market_ready=bool(model is not None and pd.notna(source.early_ret_soxx))
  risk_ready=bool(market_ready and row.portfolio_drawdown>=spec.recovery_eligibility_drawdown)
  cap_ready=bool(risk_ready and target>row.current_beta_cap+1e-12)
  alpha_ready=bool(cap_ready and pd.notna(source.alpha_probability) and source.alpha_probability>=selection['minimum_source_alpha_probability'])
  opened={'raw_beta':raw_beta,'drawdown':row.portfolio_drawdown,'current_beta_cap':row.current_beta_cap,'risk_state':row.risk_state,'history':history}
  result=prepare_on_policy_recovery(source,open_state=opened,recovery_spec=spec,model=model,selected_model=selection)
  assert bool(result['recovery_action'])==bool(row.recovery_action)
  daily.append({'trade_date':row.trade_date,'market_ready':market_ready,'risk_ready':risk_ready,'cap_ready':cap_ready,'alpha_ready':alpha_ready,'recovery_probability':result['recovery_probability'],'recovery_action':result['recovery_action'],'source_alpha':source.alpha_probability})
  previous_q=np.array([row.ending_soxl_quantity,row.ending_soxx_quantity]);previous_cash=row.ending_cash
 data=pd.DataFrame(daily);records[str(cost)]=data
 funnel.append({'slippage_bps':cost,'sessions':len(data),'market_ready':int(data.market_ready.sum()),'risk_ready':int(data.risk_ready.sum()),'cap_ready':int(data.cap_ready.sum()),'alpha_ready':int(data.alpha_ready.sum()),'alpha_veto_given_cap_ready':int((data.cap_ready&~data.alpha_ready).sum()),'recovery_actions':int(data.recovery_action.sum()),'minimum_eligible_recovery_probability':float(data.recovery_probability.min()),'maximum_eligible_recovery_probability':float(data.recovery_probability.max())})
summary={'experiment':card.name,'decision':'output_routes_verified_fixed_recovery_funnel_only','new_fits':0,'new_backtests':0,'new_orders':0,'parameter_searches':0,'objective_achieved':False,'recovery_alpha_floor':selection['minimum_source_alpha_probability'],'recovery_probability_threshold':selection['probability_threshold'],'recovery_action_beta':selection['action_beta'],'tail_used_as_direct_cash_probability':False,'tail_alpha_enter_learned_open_and_recovery_features':True,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Conditional inference on saved independent cost-path states; no changed actions, holdings or PnL. Not causal value of removing a gate, not fresh paper evidence.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='frozen_output_route_audit',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,new_orders=0)
 rec.save_objects(artifact_path='inputs',plan=plan,account_audit=account_audit,action_audit=action_audit,model_audit=model_audit,selection=selection)
 rec.save_objects(artifact_path='analysis',daily=records,funnel=pd.DataFrame(funnel),summary=summary)
 for name in ['stage1_runtime_release','stage1_open_action_value','intraday_recovery_direct_allocation']:
  rec.save_objects(artifact_path='sources',**{name:(root/'src/workflow'/f'{name}.py').read_text()})
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(funnel).to_csv(card/'recovery_funnel.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(pd.DataFrame(funnel).to_string(index=False))


# 输出实际用途：alpha门未拦截，恢复模型门从未通过

本轮0拟合、0新账户回测、0新订单、0参数搜索。追踪实际runtime_release：build_runtime_target在release存在时进入stage1_runtime_release，再调用冻结prepare_on_policy_open/recovery。验证release动作摘要哈希与恢复选择，未执行生产target或broker调用。

在该路径，现金目标由原始权重、账户状态和beta上限投影产生；tail_probability并非直接乘成现金比例或直接作为“现金胜率”阈值。tail和alpha都是开盘动作分类器及恢复分类器的普通输入。alpha另有固定0.25恢复资格门，但该量是旧混合组合标签分数，不可解释成纯SOXL净超额概率。

对已保存的两条独立成本路径，每条944个持有区间，按原账本前日数量和现金恢复known-price净值历史，并在保存的风险状态上运行冻结恢复模型。重算的恢复动作与原账本逐日一致，未改变持仓、收益或高水位；这是条件推理核查，不是移除门槛的闭环反事实。

|门|2bp|5bp|
|---|---:|---:|
|分钟信息/模型可用|944|944|
|满足8%回撤资格|514|788|
|恢复beta确能高于当前cap|228|476|
|通过alpha>=0.25|228|476|
|通过恢复模型分数>=0.825|0|0|

在前置合格日中alpha门否决数为0。冻结恢复模型分数范围2bp为[0.08675,0.71469]，5bp为[0.11631,0.71511]；最高值均低于0.825，因此没有恢复动作。action_beta固定0.6，未做任何门槛搜索。

结论：上一轮标签不同义不构成此处长期不恢复的直接原因，且没有发现tail被直接当现金胜率的误用。当前可复用历史中的直接阻断位于恢复分类器的分数门。不能据此降低门槛：低分可能是有效保守，也可能是输入/状态分布偏移或训练目标与执行域不一致；只有检查训练覆盖和特征口径后才能区分。

下一步先核查冻结恢复模型训练标准化、特征单位和当前合格日分布，尤其真实持仓状态、第一小时量价和不同历史价格面。只做输入范围与时点检查，不用事后收益挑阈值，不重训，也不把后续恢复交易必然有利作为前提。

决定：output_routes_verified_fixed_recovery_funnel_only。稳定净超额未实现，生产release未改；研究源价格/费用/时点资格继承原卡，非新鲜模拟成交证据。

[逐层门计数](recovery_funnel.csv) · [固定来源与门槛](plan.json)

Recorder `b46742c4e8de4b7f83ba8f1ddd6969b3`。
